In [ ]:
# Import necessary libraries
from sklearn.linear_model import Ridge, LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score,log_loss
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV,train_test_split ,StratifiedKFold,KFold
from sklearn.svm import SVC
from xgboost import XGBClassifier,XGBRFRegressor,XGBRegressor
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor,ExtraTreeClassifier
from sklearn.ensemble import VotingClassifier, BaggingClassifier,BaggingRegressor,RandomForestClassifier,AdaBoostClassifier,StackingClassifier,StackingRegressor
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
from sklearn.svm import SVC
from sklearn.pipeline import *
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
import os

warnings.simplefilter('ignore')
os.chdir('D:\Datasets')

conc = pd.read_csv('Concrete_Data.csv')
conc

In [ ]:
x= conc.drop('Strength',axis=1)
y=conc['Strength']

In [ ]:
# Create instances of the base regressors
lr = LinearRegression()
knn = KNeighborsRegressor()
xgbm = XGBRegressor(random_state=24)
dtc = DecisionTreeRegressor(random_state=24)

# Create the Stacking Regressor
# The base regressors are passed as a list of tuples, where the first element is the name of the regressor and the second is the regressor object
# The final_estimator is the XGBoost Regressor
# 'passthrough=True' means the input features will be passed to the final estimator
stack = StackingRegressor([('LR', lr), ('KNN', knn), ('TREE', dtc)], final_estimator=xgbm, passthrough=True)

In [ ]:
print(stack.get_params())

# Hyperparameter Tuning for Stacking Regressor

In [ ]:
# Create a K-Fold cross-validation object
# This will split the data into 5 folds, without preserving the class distribution
# 'shuffle=True' ensures the data is shuffled before splitting
# 'random_state=24' sets the random seed for reproducibility
kfold = KFold(n_splits=5, shuffle=True, random_state=24)

# Define the hyperparameter grid to search over
# The parameters are specified for the final estimator (XGBoost Regressor) and the base regressors (KNN and Decision Tree)
params = {
    'final_estimator__learning_rate': [0.1, 0.3, 0.5],
    'final_estimator__max_depth': [1, 3, 5],
    'final_estimator__n_estimators': [25, 50],
    'KNN__n_neighbors': [3, 5, 9],
    'TREE__max_depth': [None, 3]
}

# Create a GridSearchCV object
# This will perform a grid search to find the best combination of hyperparameters
# 'cv=kfold' uses the K-Fold cross-validation object
# 'verbose=3' sets the verbosity level to 3, which will print progress updates
gcv = GridSearchCV(stack, param_grid=params, cv=kfold, verbose=3)

# Fit the GridSearchCV object to the data
# This will perform the grid search and find the best hyperparameters
gcv.fit(x, y)

# Print the best parameters found by the GridSearchCV
print("Best Parameters:", gcv.best_params_)

# Print the best score (i.e., the lowest mean squared error) found during the grid search
print("Best Score:", gcv.best_score_)

# Get the best model from the GridSearchCV object
best_model = gcv.best_estimator_